### 1 - Load and inspect data

In [17]:
import pandas as pd
import numpy as np

movies = pd.read_csv("../data/ml-latest-small/movies.csv")
ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")

print("movies:", movies.shape)
print("ratings:", ratings.shape)
display(movies.head())
display(ratings.head())

print(movies.info())
print(ratings.info())

movies: (9742, 3)
ratings: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


### 2 - Basic preprocessing

In [18]:
movies["genres"] = movies["genres"].fillna("").str.replace("|", " ")
movies["metadata"] = (movies["title"].fillna(
    "") + " " + movies["genres"]).str.strip()

# keep only movies with at least one rating
rated_movie_ids = ratings["movieId"].unique()
movies = movies[movies["movieId"].isin(rated_movie_ids)].reset_index(drop=True)

# helper mappings
movieid_to_idx = {mid: idx for idx, mid in enumerate(movies["movieId"].values)}
idx_to_movieid = {idx: mid for mid, idx in movieid_to_idx.items()}

### 3 - Content-based model (TF-IDF)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(movies["metadata"].astype("str"))
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


def get_similar_movies_by_movieid(movie_id, top_n=10):
    if movie_id not in movieid_to_idx:
        return pd.DataFrame()
    idx = movieid_to_idx[movie_id]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[
        1: top_n+1]
    movie_indices = [i for i, _ in sim_scores]
    return movies.iloc[movie_indices][["movieId", "title"]]

### 4 - Split train/test (leave-one-out per user)

In [20]:
def leave_one_out_by_user(ratings_df, seed=42, min_ratings=5):
    train_parts, test_parts = [], []
    for uid, grp in ratings_df.groupby("userId"):
        if len(grp) >= min_ratings:
            test_row = grp.sample(n=1, random_state=seed)
            train_parts.append(grp.drop(test_row.index))
            test_parts.append(test_row)
        else:
            train_parts.append(grp)
    train = pd.concat(train_parts).reset_index(drop=True)
    test = pd.concat(test_parts).reset_index(drop=True)
    return train, test


train_ratings, test_ratings = leave_one_out_by_user(ratings)
print("train:", train_ratings.shape, "test:", test_ratings.shape)

train: (100226, 4) test: (610, 4)


### 5 - Collaborative Filtering (SVD)

In [21]:
from scipy.sparse.linalg import svds

# formula: R≈UΣVT (usvt)

unique_user_ids = np.sort(train_ratings["userId"].unique())
unique_movie_ids = movies["movieId"].values

user_to_idx = {uid: i for i, uid in enumerate(unique_user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(unique_movie_ids)}

n_users, n_items = len(unique_user_ids), len(unique_movie_ids)

# build ratings matrix
R = np.zeros((n_users, n_items), dtype=np.float32)
for r in train_ratings.itertuples():
    if r.movieId in movie_to_idx:
        R[user_to_idx[r.userId], movie_to_idx[r.movieId]] = r.rating

# mean-center
user_means = np.true_divide(
    R.sum(1), (R != 0).sum(1), where=(R != 0).sum(1) != 0)
R_centered = R - user_means[:, None] * (R != 0)

# train SVD
k = min(50, min(R_centered.shape) - 1)
u, s, vt = svds(R_centered, k=k)

predicted_ratings = u @ np.diag(s) @ vt + user_means[:, None]


### 6 - Evaluate

In [22]:
from sklearn.metrics import mean_squared_error

# RMSE
y_true, y_pred = [], []
for r in test_ratings.itertuples():
    if r.userId in user_to_idx and r.movieId in movie_to_idx:
        uidx, midx = user_to_idx[r.userId], movie_to_idx[r.movieId]
        y_true.append(r.rating)
        y_pred.append(predicted_ratings[uidx, midx])
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print("SVD RMSE:", rmse)

# Precision@K
def precision_at_k(pred_matrix, train_df, test_df, k=10):
    hits, total = 0, 0
    for user_id, group in test_df.groupby("userId"):
        if user_id not in user_to_idx:
            continue
        uid = user_to_idx[user_id]
        scores = pred_matrix[uid].copy()
        seen = set(train_df[train_df["userId"] == user_id]["movieId"])
        for mid in seen:
            if mid in movie_to_idx:
                scores[movie_to_idx[mid]] = -np.inf
        topk_idx = np.argsort(scores)[-k:][::-1]
        topk_movieids = set(movies.iloc[topk_idx]["movieId"])
        total += len(group)
        hits += sum(mid in topk_movieids for mid in group["movieId"])
    return hits / total if total > 0 else 0.0


p_at_10 = precision_at_k(predicted_ratings, train_ratings, test_ratings, k=10)
print("Precision@10:", p_at_10)


SVD RMSE: 0.9066197502627461
Precision@10: 0.10163934426229508


### 7 - Predict / Recommend

In [23]:
def recommend_svd_user(user_id, top_n=10):
    if user_id not in user_to_idx:
        return pd.DataFrame()
    uid = user_to_idx[user_id]
    scores = predicted_ratings[uid].copy()
    seen = set(train_ratings[train_ratings["userId"] == user_id]["movieId"])
    for mid in seen:
        if mid in movie_to_idx:
            scores[movie_to_idx[mid]] = -np.inf
    top_idx = np.argsort(scores)[-top_n:][::-1]
    return movies.iloc[top_idx][["movieId", "title"]]


print(recommend_svd_user(1, top_n=5))

      movieId                                              title
474       541                                Blade Runner (1982)
302       344                  Ace Ventura: Pet Detective (1994)
2109     2804                          Christmas Story, A (1983)
4131     5952      Lord of the Rings: The Two Towers, The (2002)
3633     4993  Lord of the Rings: The Fellowship of the Ring,...


### 8 - Save Artifacts

In [24]:
import joblib
import pathlib

path_save = pathlib.Path("models")
path_save.mkdir(exist_ok=True)
joblib.dump({
    "movieid_to_idx": movieid_to_idx,
    "idx_to_movieid": idx_to_movieid,
    "user_to_idx": user_to_idx,
    "movie_to_idx": movie_to_idx,
    "predicted_ratings": predicted_ratings,
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "movies_df": movies
}, path_save / "recommender_scikit_artifacts.pkl")

['models\\recommender_scikit_artifacts.pkl']